<a href="https://colab.research.google.com/github/sunil965/masai-aiml/blob/master/IITR_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Synthetic shipment data for the Delay Intelligence capstone.
Run once:  python generate_data.py   (writes data/hubs.csv, data/raw_shipments.csv, data/new_shipments.csv)
Do not edit any constant: every number in the brief was measured on this exact output."""
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))

24 1224 48


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
hubs_df = pd.read_csv("/content/data/hubs.csv")
hubs_df.head()

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300


In [ ]:
raw_ship_df = pd.read_csv("/content/data/raw_shipments.csv")
raw_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [ ]:
new_ship_df = pd.read_csv("/content/data/new_shipments.csv")
new_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no
0,SHP90001,HUB22,Standard,Prepaid,1.52,478.7,₹483,78,0,0,2026-08-03 17:26,9034,5,311,0.710,273
1,SHP90002,HUB24,Standard,COD,0.76,405.5,"₹1,137",94,1,0,2026-08-21 12:52,7336,8,163,0.784,284
2,SHP90003,HUB18,Standard,Prepaid,2.00,552.4,₹195,99,0,0,2026-08-07 16:29,9987,3,291,0.927,140
3,SHP90004,HUB12,Economy,Prepaid,1.66,243.0,"₹1,018",65,0,0,2026-08-28 14:41,4749,14,257,0.405,21
4,SHP90005,HUB22,Express,COD,0.72,219.1,NaN,72,0,0,2026-08-25 18:11,1364,11,442,0.832,164


In [ ]:
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
 "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value",
"n_stops",
 "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
 "hub_tier", "hub_daily_capacity",
 "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod",
"shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no",
"sticker_serial_mod", "shelf_slot_no"]

In [ ]:
PROMISED_HOURS

{'Express': 36, 'Standard': 60, 'Economy': 96}

In [ ]:
FEATURES

['promised_hours',
 'distance_km',
 'parcel_weight_kg',
 'log_declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'is_cod',
 'is_weekend',
 'dispatch_hour',
 'hub_tier',
 'hub_daily_capacity',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no']

In [ ]:
len(FEATURES)

17

In [ ]:
raw_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [ ]:
raw_ship_df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
1219,True
1220,True
1221,True
1222,True


In [ ]:
raw_ship_df.duplicated().sum()

np.int64(24)

In [ ]:
raw_ship_df.duplicated().sum()*100/len(raw_ship_df)

np.float64(1.9607843137254901)

In [ ]:
len(raw_ship_df)

1224

In [ ]:
24*100/1224

1.9607843137254901

In [ ]:
raw_ship_df.duplicated().mean()*100

np.float64(1.9607843137254901)